In [1]:
import csv
from datetime import datetime
from pathlib import Path
from statistics import mean, median, pstdev

In [2]:
BASE_DIR = Path(__file__).resolve().parent
DATA_FILES = list(BASE_DIR.rglob("ventas_mensuales.csv"))

if not DATA_FILES:
    raise FileNotFoundError("No se encontro ventas_mensuales.csv dentro de la carpeta abierta")

DATA_FILE = DATA_FILES[0]
OUTPUT_FILE = BASE_DIR / "ventas_mensuales_tratada.csv"

TEXT_COLUMNS = {
    "periodo",
    "region",
    "sucursal",
    "categoria",
    "producto",
    "laboratorio",
}
NUMERIC_COLUMNS = {
    "anio",
    "mes",
    "cantidad",
    "precio",
    "descuento_pct",
    "ventas",
}

with DATA_FILE.open("r", encoding="utf-8-sig", newline="") as source:
    reader = csv.DictReader(source)
    fieldnames = reader.fieldnames or []
    source_rows = list(reader)

if not fieldnames:
    raise ValueError("La base no contiene encabezados")

clean_rows = []
seen = set()
duplicate_count = 0
invalid_period_count = 0

for row in source_rows:
    clean = dict(row)

    for column in TEXT_COLUMNS & set(fieldnames):
        value = clean.get(column)
        clean[column] = value.strip() if value else ""

    for column in NUMERIC_COLUMNS & set(fieldnames):
        try:
            clean[column] = float(clean[column])
        except (TypeError, ValueError):
            clean[column] = None

    periodo = clean.get("periodo")
    try:
        datetime.strptime(periodo, "%Y-%m")
    except (TypeError, ValueError):
        invalid_period_count += 1
        continue

    signature = tuple(clean[column] for column in fieldnames)
    if signature in seen:
        duplicate_count += 1
        continue

    seen.add(signature)
    clean_rows.append(clean)

with OUTPUT_FILE.open("w", encoding="utf-8-sig", newline="") as target:
    writer = csv.DictWriter(target, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(clean_rows)

print("ARCHIVO FUENTE:")
print(DATA_FILE)
print(f"\nARCHIVO TRATADO: {OUTPUT_FILE}")
print(f"\nDimensiones: {len(clean_rows):,} filas x {len(fieldnames)} columnas")
print(f"Filas duplicadas exactas eliminadas: {duplicate_count:,}")
print(f"Filas con periodo invalido eliminadas: {invalid_period_count:,}")

periods = [row["periodo"] for row in clean_rows if row.get("periodo")]
if periods:
    print(f"Periodo disponible: {min(periods)} a {max(periods)}")

missing = {
    column: sum(1 for row in clean_rows if row.get(column) in ("", None))
    for column in fieldnames
}
missing = {column: count for column, count in missing.items() if count}
print("\nVALORES AUSENTES:")
print("\n".join(f"{column}: {count:,}" for column, count in missing.items()) or "Ninguno")

print("\nDIMENSIONES PRINCIPALES:")
for column in ("region", "sucursal", "categoria", "producto", "laboratorio"):
    if column in fieldnames:
        values = {row.get(column) for row in clean_rows if row.get(column) not in ("", None)}
        print(f"{column}: {len(values):,} valores")

print("\nRESUMEN NUMERICO:")
for column in NUMERIC_COLUMNS & set(fieldnames):
    values = [
        row[column]
        for row in clean_rows
        if isinstance(row.get(column), (int, float))
    ]
    if values:
        print(
            f"{column}: min={min(values):,.2f}, mediana={median(values):,.2f}, "
            f"promedio={mean(values):,.2f}, desviacion={pstdev(values):,.2f}, "
            f"max={max(values):,.2f}, total={sum(values):,.2f}"
        )


def summarize(values):
    if not values:
        return None
    return {
        "count": len(values),
        "total": sum(values),
        "mean": mean(values),
        "median": median(values),
        "min": min(values),
        "max": max(values),
        "std": pstdev(values),
    }


def write_group_report(column, output_name):
    groups = {}
    for row in clean_rows:
        key = row.get(column)
        if key in ("", None):
            continue
        aggregate = groups.setdefault(
            key,
            {"ventas": [], "cantidad": [], "filas": 0},
        )
        aggregate["filas"] += 1
        aggregate["ventas"].append(row["ventas"])
        aggregate["cantidad"].append(row["cantidad"])

    total_sales = sum(row["ventas"] for row in clean_rows)
    output_path = BASE_DIR / output_name
    with output_path.open("w", encoding="utf-8-sig", newline="") as target:
        writer = csv.DictWriter(
            target,
            fieldnames=[column, "filas", "ventas", "cantidad", "participacion_ventas"],
        )
        writer.writeheader()
        for key, aggregate in sorted(
            groups.items(),
            key=lambda item: sum(item[1]["ventas"]),
            reverse=True,
        ):
            sales = sum(aggregate["ventas"])
            writer.writerow(
                {
                    column: key,
                    "filas": aggregate["filas"],
                    "ventas": round(sales, 2),
                    "cantidad": round(sum(aggregate["cantidad"]), 2),
                    "participacion_ventas": round(sales / total_sales * 100, 2)
                    if total_sales
                    else 0,
                }
            )
    return output_path


def write_period_report(column, output_name):
    periods = {}
    for row in clean_rows:
        key = str(row.get(column))
        aggregate = periods.setdefault(key, {"ventas": [], "cantidad": [], "filas": 0})
        aggregate["filas"] += 1
        aggregate["ventas"].append(row["ventas"])
        aggregate["cantidad"].append(row["cantidad"])

    output_path = BASE_DIR / output_name
    with output_path.open("w", encoding="utf-8-sig", newline="") as target:
        writer = csv.DictWriter(
            target,
            fieldnames=[column, "filas", "ventas", "cantidad", "ventas_promedio"],
        )
        writer.writeheader()
        for key in sorted(periods):
            aggregate = periods[key]
            writer.writerow(
                {
                    column: key,
                    "filas": aggregate["filas"],
                    "ventas": round(sum(aggregate["ventas"]), 2),
                    "cantidad": round(sum(aggregate["cantidad"]), 2),
                    "ventas_promedio": round(mean(aggregate["ventas"]), 2),
                }
            )
    return output_path


def pearson(x_values, y_values):
    if len(x_values) != len(y_values) or len(x_values) < 2:
        return None
    x_mean = mean(x_values)
    y_mean = mean(y_values)
    numerator = sum(
        (x - x_mean) * (y - y_mean)
        for x, y in zip(x_values, y_values)
    )
    x_variance = sum((x - x_mean) ** 2 for x in x_values)
    y_variance = sum((y - y_mean) ** 2 for y in y_values)
    denominator = (x_variance * y_variance) ** 0.5
    return numerator / denominator if denominator else None


analysis_dir = BASE_DIR / "analisis_descriptivo"
analysis_dir.mkdir(exist_ok=True)


def analysis_group(column, output_name):
    groups = {}
    for row in clean_rows:
        key = row.get(column)
        if key in ("", None):
            continue
        aggregate = groups.setdefault(key, {"ventas": [], "cantidad": [], "filas": 0})
        aggregate["filas"] += 1
        aggregate["ventas"].append(row["ventas"])
        aggregate["cantidad"].append(row["cantidad"])

    total_sales = sum(row["ventas"] for row in clean_rows)
    output_path = analysis_dir / output_name
    with output_path.open("w", encoding="utf-8-sig", newline="") as target:
        writer = csv.DictWriter(
            target,
            fieldnames=[column, "filas", "ventas", "cantidad", "participacion_ventas"],
        )
        writer.writeheader()
        for key, aggregate in sorted(
            groups.items(),
            key=lambda item: sum(item[1]["ventas"]),
            reverse=True,
        ):
            sales = sum(aggregate["ventas"])
            writer.writerow(
                {
                    column: key,
                    "filas": aggregate["filas"],
                    "ventas": round(sales, 2),
                    "cantidad": round(sum(aggregate["cantidad"]), 2),
                    "participacion_ventas": round(sales / total_sales * 100, 2)
                    if total_sales
                    else 0,
                }
            )
    return output_path


def analysis_period(column, output_name):
    periods = {}
    for row in clean_rows:
        key = str(row.get(column))
        aggregate = periods.setdefault(key, {"ventas": [], "cantidad": [], "filas": 0})
        aggregate["filas"] += 1
        aggregate["ventas"].append(row["ventas"])
        aggregate["cantidad"].append(row["cantidad"])

    output_path = analysis_dir / output_name
    with output_path.open("w", encoding="utf-8-sig", newline="") as target:
        writer = csv.DictWriter(
            target,
            fieldnames=[column, "filas", "ventas", "cantidad", "ventas_promedio"],
        )
        writer.writeheader()
        for key in sorted(periods):
            aggregate = periods[key]
            writer.writerow(
                {
                    column: key,
                    "filas": aggregate["filas"],
                    "ventas": round(sum(aggregate["ventas"]), 2),
                    "cantidad": round(sum(aggregate["cantidad"]), 2),
                    "ventas_promedio": round(mean(aggregate["ventas"]), 2),
                }
            )
    return output_path


NameError: name '__file__' is not defined

In [3]:
numeric_summaries = {}
for column in NUMERIC_COLUMNS & set(fieldnames):
    values = [
        row[column]
        for row in clean_rows
        if isinstance(row.get(column), (int, float))
    ]
    numeric_summaries[column] = summarize(values)

analysis_region = analysis_group("region", "ventas_por_region.csv")
analysis_category = analysis_group("categoria", "ventas_por_categoria.csv")
analysis_product = analysis_group("producto", "ventas_por_producto.csv")
analysis_lab = analysis_group("laboratorio", "ventas_por_laboratorio.csv")
analysis_branch = analysis_group("sucursal", "ventas_por_sucursal.csv")
analysis_month = analysis_period("periodo", "ventas_mensuales.csv")
analysis_year = analysis_period("anio", "ventas_por_anio.csv")

cantidad = [row["cantidad"] for row in clean_rows]
precio = [row["precio"] for row in clean_rows]
ventas = [row["ventas"] for row in clean_rows]
descuento = [row["descuento_pct"] for row in clean_rows]

report_lines = [
    "ANALISIS DESCRIPTIVO DE VENTAS",
    "=" * 34,
    "",
    "1. COBERTURA",
    f"- Fuente: {DATA_FILE}",
    f"- Registros analizados: {len(clean_rows):,}",
    f"- Periodo: {min(periods)} a {max(periods)}",
    f"- Regiones: {len({row['region'] for row in clean_rows})}",
    f"- Sucursales: {len({row['sucursal'] for row in clean_rows})}",
    f"- Categorias: {len({row['categoria'] for row in clean_rows})}",
    f"- Productos: {len({row['producto'] for row in clean_rows})}",
    f"- Laboratorios: {len({row['laboratorio'] for row in clean_rows})}",
    "",
    "2. RESULTADOS GENERALES",
]

for column in ("ventas", "cantidad", "precio", "descuento_pct"):
    summary = numeric_summaries[column]
    report_lines.append(
        f"- {column}: total={summary['total']:,.2f}; "
        f"promedio={summary['mean']:,.2f}; mediana={summary['median']:,.2f}; "
        f"desviacion={summary['std']:,.2f}; "
        f"min={summary['min']:,.2f}; max={summary['max']:,.2f}"
    )

def top_entries(path, label_column, limit=5):
    with path.open("r", encoding="utf-8-sig", newline="") as source:
        rows = list(csv.DictReader(source))
    return rows[:limit], label_column

report_lines.extend(["", "3. PRINCIPALES SEGMENTOS POR VENTAS"])
for title, path, key_column in (
    ("Regiones", analysis_region, "region"),
    ("Categorias", analysis_category, "categoria"),
    ("Productos", analysis_product, "producto"),
    ("Laboratorios", analysis_lab, "laboratorio"),
    ("Sucursales", analysis_branch, "sucursal"),
):
    top_rows, _ = top_entries(path, key_column, 5)
    report_lines.append(f"- {title}:")
    for rank, row in enumerate(top_rows, 1):
        report_lines.append(
            f"  {rank}. {row[key_column]}: {float(row['ventas']):,.2f} "
            f"({float(row['participacion_ventas']):.2f}% del total)"
        )

correlation_pairs = (
    ("cantidad y ventas", cantidad, ventas),
    ("precio y ventas", precio, ventas),
    ("descuento y ventas", descuento, ventas),
)
report_lines.extend(["", "4. RELACIONES LINEALES (CORRELACION DE PEARSON)"])
for label, x_values, y_values in correlation_pairs:
    correlation = pearson(x_values, y_values)
    report_lines.append(
        f"- {label}: {correlation:.4f}"
        if correlation is not None
        else f"- {label}: no calculable"
    )

with (analysis_dir / "informe_descriptivo.txt").open(
    "w",
    encoding="utf-8",
) as report:
    report.write("\n".join(report_lines))

print("\nANALISIS DESCRIPTIVO:")
print("\n".join(report_lines))
print(f"\nArchivos de analisis: {analysis_dir}")

NameError: name 'NUMERIC_COLUMNS' is not defined